# Kvasir-SEG Inference Lab

This notebook loads the trained U-Net checkpoint, runs inference on one Kvasir-SEG image, and saves a predicted segmentation mask. It also includes a few short questions to help you inspect GPU usage and inference performance.

## 1. Setup

Run this notebook from the project root or from the `notebooks/` folder. The checkpoint is expected at `outputs/checkpoints/unet_kvasir.pt`.

In [ ]:
from pathlib import Path
import subprocess
import sys
import time
import zipfile

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

from inference import load_model, predict_mask, save_prediction, show_prediction

checkpoint_path = PROJECT_ROOT / "outputs/checkpoints/unet_kvasir.pt"
image_path = PROJECT_ROOT / "data/Kvasir-SEG/images/cju0qkwl35piu0993l0dewei2.jpg"
output_path = PROJECT_ROOT / "outputs/predictions/notebook_prediction.png"

if not image_path.exists():
    zip_path = PROJECT_ROOT / "data/kvasir-seg.zip"
    sample_dir = PROJECT_ROOT / "outputs/predictions"
    sample_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        image_name = next(
            name for name in archive.namelist()
            if "/images/" in name and Path(name).suffix.lower() in {".jpg", ".jpeg", ".png"}
        )
        image_path = sample_dir / Path(image_name).name
        image_path.write_bytes(archive.read(image_name))

checkpoint_path, image_path, output_path

## 2. Check GPU Availability

Question 1: Is the notebook using a GPU? Run the cell below and write down the GPU name and the initial memory usage.

In [ ]:
try:
    result = subprocess.run(["nvidia-smi"], check=False, text=True, capture_output=True)
    print(result.stdout if result.stdout else result.stderr)
except FileNotFoundError:
    print("nvidia-smi is not available in this environment.")

## 3. Load the Model

Question 2: Which device does PyTorch select for inference: `cuda` or `cpu`?

In [ ]:
model, device = load_model(checkpoint_path)
print(f"Inference device: {device}")

## 4. Run Inference

Question 3: Measure the inference time for one image. Is it below one second? What changes if you run the same cell twice?

In [ ]:
start = time.perf_counter()
probability, mask = predict_mask(model, image_path, device, image_size=256, threshold=0.5)
elapsed = time.perf_counter() - start

save_prediction(mask, output_path)
fig = show_prediction(image_path, probability, mask)

print(f"Inference time: {elapsed:.4f} seconds")
print(f"Saved prediction to {output_path}")

## 5. Explore the Threshold

Question 4: Change `threshold` to `0.3`, `0.5`, and `0.7`. How does the predicted mask change?

In [ ]:
threshold = 0.5
probability, mask = predict_mask(model, image_path, device, image_size=256, threshold=threshold)
fig = show_prediction(image_path, probability, mask)
print(f"Threshold: {threshold}")

## 6. Small Challenge

Propose one extra measurement for inference. Examples:

- Average inference time over 10 runs.
- GPU memory before and after inference.
- Number of foreground pixels in the predicted mask.
- Comparison between CPU and GPU inference time.